# Modellierung – Training und Vorhersagen

Wir prognostizieren monatliche Pending Listings für ein bis zwölf Monate voraus. Dafür trainieren wir 24 Modellkandidaten auf den vorbereiteten Daten. Die Bewertung erfolgt im nächsten Notebook.

## 1. Bibliotheken und Datenzugriff

die 13 vorbereiteten CSV-Dateien werden geladen . Die Ausgaben werden unter `ergebnisse/modellierung` gespeichert.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import sklearn
import statsmodels

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX


project_dir = Path(
    r"C:\Users\mmatavela\Desktop\House_Demand_Forecasting"
)

# Falls nötig: den Ordner mit den vorbereiteten Dateien direkt eintragen.
prepared_data_dir = None

required_files = ["monthly_base.csv"] + [
    f"supervised_h{horizon:02d}_unscaled.csv"
    for horizon in range(1, 13)
]

if prepared_data_dir is None:
    candidate_dirs = set()

    for folder_name in ["ergebnisse", "outputs", "daten"]:
        search_dir = project_dir / folder_name

        if search_dir.is_dir():
            for monthly_file in search_dir.rglob("monthly_base.csv"):
                files_complete = all(
                    (monthly_file.parent / file_name).is_file()
                    for file_name in required_files
                )

                if files_complete:
                    candidate_dirs.add(monthly_file.parent.resolve())

    if not candidate_dirs:
        raise FileNotFoundError(
            "CSV-Dateien wurden nicht gefunden. "
            "Ausgaben aus Data Preparation prüfen."
        )

    if len(candidate_dirs) > 1:
        raise ValueError(
            "Many similar data directories found. "
            "prepared_data_dir in the right Folder."
        )

    prepared_data_dir = next(iter(candidate_dirs))

data_dir = Path(prepared_data_dir)

missing_files = [
    file_name
    for file_name in required_files
    if not (data_dir / file_name).is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Fehlende Dateien: " + ", ".join(missing_files)
    )

results_dir = project_dir / "ergebnisse" / "modellierung"
results_dir.mkdir(parents=True, exist_ok=True)

print("Bibliotheken geladen und alle 13 Eingabedateien gefunden.")

Bibliotheken geladen und alle 13 Eingabedateien gefunden.


## 2. Daten, Features und Prognosezeiträume

Wir laden die Monatsdaten und bilden die Feature-Gruppen Minimal, Basis und Permits. Die jährliche Validierung umfasst August-Starts 2021–2024, die monatliche Ergänzung Januar 2023–August 2024.

Der bekannte Test startet im August 2025. Für Zielwerte gilt eine angenommene Datenverzögerung von zwei Monaten.

In [3]:
monthly_data = pd.read_csv(
    data_dir / "monthly_base.csv",
    index_col="month"
)

monthly_data.index = pd.PeriodIndex(
    monthly_data.index,
    freq="M"
)

monthly_data = monthly_data.sort_index()

horizon_tables = {}

for horizon in range(1, 13):
    horizon_tables[horizon] = pd.read_csv(
        data_dir / f"supervised_h{horizon:02d}_unscaled.csv"
    )

all_features = [
    column
    for column in horizon_tables[1].columns
    if "_lag_" in column
] + ["month_sin", "month_cos"]

feature_groups = {
    "Minimal": [
        "pending_lag_2",
        "pending_lag_12",
        "month_sin",
        "month_cos"
    ],
    "Basis": [
        column
        for column in all_features
        if column != "permits_lag_2"
    ],
    "Permits": all_features
}

annual_origins = pd.PeriodIndex(
    ["2021-08", "2022-08", "2023-08", "2024-08"],
    freq="M"
)

monthly_origins = pd.period_range(
    "2023-01", "2024-08", freq="M"
)

test_origin = pd.Period("2025-08", freq="M")

target_delay = 2
forecast_horizons = range(1, 13)
ridge_alphas = [0.1, 1, 10, 100, 1000]

assert monthly_data.index.is_unique
assert np.isfinite(monthly_data["pending"]).all()

print(f"Monatsdaten: {len(monthly_data)} Beobachtungen")
print(f"Jährliche Prognosestarts: {len(annual_origins)}")
print(f"Monatliche Prognosestarts: {len(monthly_origins)}")

Monatsdaten: 108 Beobachtungen
Jährliche Prognosestarts: 4
Monatliche Prognosestarts: 20


## 3. ETS und SARIMA trainieren

Wir trainieren drei ETS-Varianten und SARIMA auf der verfügbaren Pending-Historie. Wegen der zweimonatigen Datenverzögerung werden 14 Schritte für die zwölf Zielmonate berechnet. Warnungen und Konvergenz werden geprüft.

In [4]:
def fit_time_series_models(target_history):
    model_forecasts = {}
    fit_logs = []
    forecast_steps = 12 + target_delay

    ets_variants = [
        ("ETS ohne Trend", None, False),
        ("ETS Trend", "add", False),
        ("ETS gedämpft", "add", True)
    ]

    for model_name, trend, damped_trend in ets_variants:
        with warnings.catch_warnings(record=True) as model_warnings:
            model = ExponentialSmoothing(
                target_history.to_numpy(),
                trend=trend,
                damped_trend=damped_trend,
                seasonal="add",
                seasonal_periods=12,
                initialization_method="estimated"
            )
            fitted_model = model.fit(optimized=True, use_brute=True)

        converged = bool(fitted_model.mle_retvals["success"])
        if not converged:
            raise RuntimeError(
                f"Die Parameterschätzung für {model_name} "
                "wurde nicht erfolgreich abgeschlossen."
            )

        model_forecasts[model_name] = fitted_model.forecast(forecast_steps)
        fit_logs.append({
            "model": model_name,
            "converged": converged,
            "warnings": [str(warning.message) for warning in model_warnings]
        })

    with warnings.catch_warnings(record=True) as model_warnings:
        sarima_model = SARIMAX(
            target_history.to_numpy(),
            order=(1, 0, 0),
            seasonal_order=(1, 1, 0, 12),
            trend="n",
            enforce_stationarity=True,
            enforce_invertibility=True
        )
        fitted_model = sarima_model.fit(disp=False, maxiter=500)

        if not fitted_model.mle_retvals["converged"]:
            fitted_model = sarima_model.fit(
                disp=False, method="powell", maxiter=1000
            )

    converged = bool(fitted_model.mle_retvals["converged"])
    if not converged:
        raise RuntimeError(
            "Die Parameterschätzung für SARIMA "
            "wurde nicht erfolgreich abgeschlossen."
        )

    model_forecasts["SARIMA"] = fitted_model.forecast(forecast_steps)
    fit_logs.append({
        "model": "SARIMA",
        "converged": converged,
        "warnings": [str(warning.message) for warning in model_warnings]
    })
    return model_forecasts, fit_logs

## 4. Vorhersagen an einem Prognosestart erstellen

Wir berechnen die Referenzprognosen und trainieren Ridge sowie Gradient Boosting getrennt je Horizont. Ridge standardisiert die Lag Features ausschließlich anhand der Trainingsdaten; Boosting nutzt unskalierte Werte.

Gespeichert werden Prognosen, tatsächliche Werte und Fehler einschließlich der Skalierung für MASE. Zukünftige Zielwerte dienen nur der Bewertung.

In [ ]:
def train_at_origin(forecast_origin):
    prediction_rows = []
    last_available_month = forecast_origin - target_delay
    target_history = monthly_data.loc[:last_available_month, "pending"]
    history_values = target_history.to_numpy()

    seasonal_scale = np.abs(
        history_values[12:] - history_values[:-12]
    ).mean()

    if not np.isfinite(seasonal_scale) or seasonal_scale <= 0:
        raise ValueError(
            "Die historische Vergleichsskala für MASE ist nicht gültig."
        )

    time_series_forecasts, fit_logs = fit_time_series_models(target_history)
    for fit_log in fit_logs:
        fit_log["origin"] = str(forecast_origin)

    for horizon in forecast_horizons:
        target_month = forecast_origin + horizon
        actual_value = float(monthly_data.loc[target_month, "pending"])

        #Jüngster verfügbarer gleicher Kalendermonat.
        seasonal_month = target_month - 12
        while seasonal_month > last_available_month:
            seasonal_month -= 12

        seasonal_values = [
            float(target_history.loc[seasonal_month - 12 * year])
            for year in range(3)
        ]
        predictions = {
            "Naive": float(target_history.iloc[-1]),
            "Seasonal Naive": seasonal_values[0],
            "Seasonal Mean 3Y": float(np.mean(seasonal_values))
        }

        # Verzögerung und nullbasierte Indizierung berücksichtigen.
        forecast_position = target_delay + horizon - 1
        for model_name, forecast_values in time_series_forecasts.items():
            predictions[model_name] = float(forecast_values[forecast_position])

        horizon_data = horizon_tables[horizon]
        training_data = horizon_data.loc[
            horizon_data["target_available_month_assumed"] <= str(forecast_origin)
        ]
        forecast_data = horizon_data.loc[
            horizon_data["origin_month"] == str(forecast_origin)
        ]

        assert not training_data.empty
        assert len(forecast_data) == 1
        assert training_data["target_month"].max() <= str(last_available_month)
        assert training_data["origin_month"].max() < str(forecast_origin)
        assert float(forecast_data["target_pending"].iloc[0]) == actual_value

        training_target = training_data["target_pending"]

        for group_name, feature_columns in feature_groups.items():
            lag_columns = [
                column for column in feature_columns if "_lag_" in column
            ]
            training_features = training_data[feature_columns].copy()
            forecast_features = forecast_data[feature_columns].copy()

            # Skalierung ausschließlich mit dem aktuellen Trainingssatz.
            scaler = StandardScaler()
            scaler.fit(training_data[lag_columns])
            training_features[lag_columns] = scaler.transform(
                training_data[lag_columns]
            )
            forecast_features[lag_columns] = scaler.transform(
                forecast_data[lag_columns]
            )

            for alpha in ridge_alphas:
                ridge_model = Ridge(alpha=alpha)
                ridge_model.fit(training_features, training_target)
                model_name = f"Ridge {group_name} a={alpha:g}"
                predictions[model_name] = float(
                    ridge_model.predict(forecast_features)[0]
                )

            if group_name in ["Basis", "Permits"]:
                boosting_model = GradientBoostingRegressor(
                    n_estimators=80,
                    learning_rate=0.05,
                    max_depth=2,
                    min_samples_leaf=8,
                    random_state=42
                )
                boosting_model.fit(
                    training_data[feature_columns], training_target
                )
                predictions[f"GB {group_name}"] = float(
                    boosting_model.predict(forecast_data[feature_columns])[0]
                )

        for model_name, predicted_value in predictions.items():
            forecast_error = predicted_value - actual_value
            prediction_rows.append({
                "origin": str(forecast_origin),
                "target": str(target_month),
                "horizon": horizon,
                "model": model_name,
                "actual": actual_value,
                "prediction": predicted_value,
                "error": forecast_error,
                "ae": abs(forecast_error),
                "scaled_error": abs(forecast_error) / seasonal_scale,
                "train_n": len(training_data)
            })

    return prediction_rows, fit_logs

## 5. Training durchführen und Vorhersagen speichern

Wir führen alle Prognosestarts aus und prüfen, ob je Start und Modell zwölf Vorhersagen vorliegen. Einzelprognosen, Trainingsprotokoll und Einstellungen werden für die weitere Auswertung gespeichert.

In [6]:
forecast_origins = sorted(
    set(annual_origins) | set(monthly_origins) | {test_origin}
)
prediction_rows = []
fit_logs = []

for start_number, forecast_origin in enumerate(forecast_origins, start=1):
    origin_rows, origin_logs = train_at_origin(forecast_origin)
    prediction_rows.extend(origin_rows)
    fit_logs.extend(origin_logs)
    print(
        f"{start_number}/{len(forecast_origins)} "
        f"Prognosestarts abgeschlossen: {forecast_origin}"
    )

predictions_data = pd.DataFrame(prediction_rows)
assert np.isfinite(predictions_data["prediction"]).all()
assert predictions_data["model"].nunique() == 24
assert not predictions_data.duplicated(["origin", "horizon", "model"]).any()

predictions_per_model = predictions_data.groupby(["origin", "model"]).size()
assert predictions_per_model.eq(12).all()

annual_predictions = predictions_data.loc[
    predictions_data["origin"].isin(annual_origins.astype(str))
]
monthly_predictions = predictions_data.loc[
    predictions_data["origin"].isin(monthly_origins.astype(str))
]
test_predictions = predictions_data.loc[
    predictions_data["origin"] == str(test_origin)
]
assert annual_predictions.groupby("model").size().eq(48).all()
assert monthly_predictions.groupby("model").size().eq(240).all()
assert test_predictions.groupby("model").size().eq(12).all()

predictions_data.to_csv(results_dir / "all_predictions.csv", index=False)
(results_dir / "fit_log.json").write_text(
    json.dumps(fit_logs, ensure_ascii=False, indent=2), encoding="utf-8"
)

run_protocol = {
    "target": "pending",
    "target_delay_months": target_delay,
    "forecast_horizons": list(forecast_horizons),
    "annual_origins": annual_origins.astype(str).tolist(),
    "monthly_origins": monthly_origins.astype(str).tolist(),
    "test_origin": str(test_origin),
    "test_status": "Bereits bekannt; keine unabhängige Bestätigung",
    "feature_groups": feature_groups,
    "ridge_alphas": ridge_alphas,
    "sarima_order": [1, 0, 0],
    "sarima_seasonal_order": [1, 1, 0, 12],
    "gradient_boosting": {
        "n_estimators": 80,
        "learning_rate": 0.05,
        "max_depth": 2,
        "min_samples_leaf": 8,
        "random_state": 42
    },
    "versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "statsmodels": statsmodels.__version__
    }
}
(results_dir / "run_protocol.json").write_text(
    json.dumps(run_protocol, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(
    f"\nTraining abgeschlossen: "
    f"{predictions_data['model'].nunique()} Modellkandidaten, "
    f"{len(predictions_data):,} Einzelprognosen."
)

1/23 Prognosestarts abgeschlossen: 2021-08
2/23 Prognosestarts abgeschlossen: 2022-08
3/23 Prognosestarts abgeschlossen: 2023-01
4/23 Prognosestarts abgeschlossen: 2023-02
5/23 Prognosestarts abgeschlossen: 2023-03
6/23 Prognosestarts abgeschlossen: 2023-04
7/23 Prognosestarts abgeschlossen: 2023-05
8/23 Prognosestarts abgeschlossen: 2023-06
9/23 Prognosestarts abgeschlossen: 2023-07
10/23 Prognosestarts abgeschlossen: 2023-08
11/23 Prognosestarts abgeschlossen: 2023-09
12/23 Prognosestarts abgeschlossen: 2023-10
13/23 Prognosestarts abgeschlossen: 2023-11
14/23 Prognosestarts abgeschlossen: 2023-12
15/23 Prognosestarts abgeschlossen: 2024-01
16/23 Prognosestarts abgeschlossen: 2024-02
17/23 Prognosestarts abgeschlossen: 2024-03
18/23 Prognosestarts abgeschlossen: 2024-04
19/23 Prognosestarts abgeschlossen: 2024-05
20/23 Prognosestarts abgeschlossen: 2024-06
21/23 Prognosestarts abgeschlossen: 2024-07
22/23 Prognosestarts abgeschlossen: 2024-08
23/23 Prognosestarts abgeschlossen: 2025-